# COMP5318 Assignment 1: Rice Classification — No-Leakage VersionThis notebook is a **standalone, supplementary** version of the assignment pipeline. It reproduces the same classifiers and reports the same metrics as the submitted notebook, but avoids data leakage between train and test data at every step:- **Part 1** (cross-validation only): the imputer and scaler are refit inside *every* CV fold, via a `Pipeline`, instead of once on the full dataset.- **Part 2** (train/test split + tuning): the imputer and scaler are fit on the **training split only**, then applied (transform-only) to the test split.This is run locally to quantify how much the leakage in the main submitted notebook actually changes the results (see the Reflection section there).

##### Group number: 54##### Student 1 SID: 550344487##### Student 2 SID: 550894775

## 1. Data Pre-processing (raw, unprocessed)

In [1]:
# Import all libraries
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np

In [2]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [3]:
# Load the rice dataset: rice-final2.csv
df = pd.read_csv('rice-final2.csv')
col = df.columns
class_col = col[-1]
print(df.head(5))

    Area    Perimiter Major_Axis_Length  ... Convex_Area       Extent   class
0  12573  461.4660034       192.9033508  ...       12893  0.550433397  class2
1  12845  464.1210022       194.3322144  ...       13125  0.774962306  class2
2  14055  488.7489929       207.7517548  ...       14484  0.550076306  class1
3  14412  490.3240051       207.4761353  ...       14703  0.598853171  class1
4  14658  477.1170044       189.5666351  ...       15048  0.649503708  class2

[5 rows x 8 columns]


In [4]:
# Check how many missing values ("?") in each column
for i in col:
    n_missing = df[df[i] == "?"][i].count()
    print(f'Missing value count in {i}: {n_missing}')

Missing value count in Area: 4
Missing value count in Perimiter: 4
Missing value count in Major_Axis_Length: 5
Missing value count in Minor_Axis_Length: 3
Missing value count in Eccentricity: 6
Missing value count in Convex_Area: 5
Missing value count in Extent: 2
Missing value count in class: 0


In [5]:
# Build the RAW feature matrix and label vector.
# IMPORTANT: unlike the leaky version, we do NOT impute or scale here.
# '?' is converted to NaN so SimpleImputer can recognise it, but the
# imputer/scaler themselves are only ever fit later, inside a fold or on
# the training split -- never on data that includes the test fold/split.
exe_df = df.drop(columns=class_col).replace('?', np.nan).astype(float)
y = df[class_col].map({'class1': 0, 'class2': 1}).astype(int).values

print("Raw feature matrix shape:", exe_df.shape)
print("Label vector shape:", y.shape)
print("Class balance -> class1 (0):", (y == 0).sum(), " class2 (1):", (y == 1).sum())

Raw feature matrix shape: (1400, 7)
Label vector shape: (1400,)
Class balance -> class1 (0): 600  class2 (1): 800


### Preview of the pre-processed dataset (for spec-format reference only)The assignment spec asks for a printed preview of the pre-processed data. This preview below fits the imputer/scaler on the **full** dataset purely to produce the required printout — it is a one-off display, not used anywhere in the modelling below. Every classifier further down refits its own imputer/scaler in a leakage-free way.

In [6]:
def print_data(X, y, n_rows=10):
    """Prints the first n_rows of X (4 dp) and y, matching the assignment format."""
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")
        if example_num == len(X) - 1:
            print(y[example_num], end="")
        else:
            print(y[example_num])

_preview_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', MinMaxScaler()),
])
_preview_X = _preview_pipe.fit_transform(exe_df)
print_data(_preview_X, y)

0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


## 2. Build Classifiers (Leakage-Free)- Part 1: Logistic Regression, Naive Bayes- Part 2: KNN, Decision Tree, AdaBoost, Gradient Boosting, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning (leakage-free)Each classifier is wrapped in a `Pipeline` with its own `SimpleImputer` and `MinMaxScaler`. `cross_val_score` refits this pipeline from scratch on the 9 training folds of every split, so the held-out fold never contributes to the imputation mean or the scaling min/max.

In [7]:
## Setting the 10 fold stratified cross-validation
cvKFold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

def make_pipe(clf):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', MinMaxScaler()),
        ('clf', clf),
    ])

In [8]:
# Logistic Regression
logreg_pipe = make_pipe(LogisticRegression(random_state=0))
logreg_scores = cross_val_score(logreg_pipe, exe_df, y, cv=cvKFold)
logreg_acc = logreg_scores.mean()

In [9]:
# Naive Bayes
nb_pipe = make_pipe(GaussianNB())
nb_scores = cross_val_score(nb_pipe, exe_df, y, cv=cvKFold)
nb_acc = nb_scores.mean()

### Part 1 Results

In [10]:
print("LogR average cross-validation accuracy: {:.4f}".format(logreg_acc))
print("NB average cross-validation accuracy: {:.4f}".format(nb_acc))

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264


### Part 2: Cross-validation with parameter tuning (leakage-free)Pipeline:1. Split the **raw** (unimputed, unscaled) features `exe_df`/`y` once with `train_test_split` (`stratify=y`, `random_state=0`).2. Fit `SimpleImputer` and `MinMaxScaler` on the **training split only**; `transform` (never `fit`) the test split with those same fitted objects.3. Tune each classifier on the processed training data with `GridSearchCV(..., cv=cvKFold)`.4. Report best params, best CV accuracy (`best_score_`), and test accuracy. Random Forest also reports test macro F1 and weighted F1.

In [11]:
# Split RAW data first (before any imputation/scaling), matching the leaky
# version's random_state=0 so the row split itself is identical -- only the
# preprocessing statistics differ.
Xr_train, Xr_test, y_train, y_test = train_test_split(
    exe_df, y, stratify=y, random_state=0
)

# Fit imputer + scaler on TRAIN ONLY, then transform both splits.
imputer = SimpleImputer(strategy='mean')
X_train_imp = imputer.fit_transform(Xr_train)
X_test_imp = imputer.transform(Xr_test)          # transform only, never fit on test

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_imp)
X_test = scaler.transform(X_test_imp)            # transform only, never fit on test

In [12]:
# KNN
k = [1, 3, 5, 7]
p = [1, 2]
knn_param_grid = {'n_neighbors': k, 'p': p}

knn_grid = GridSearchCV(KNeighborsClassifier(), knn_param_grid, cv=cvKFold)
knn_grid.fit(X_train, y_train)
knn_best_k = knn_grid.best_params_['n_neighbors']
knn_best_p = knn_grid.best_params_['p']
knn_cv_acc = knn_grid.best_score_
knn_test_acc = knn_grid.score(X_test, y_test)

In [13]:
# Decision Tree
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]
dt_param_grid = {
    'max_depth': max_depth,
    'min_samples_split': min_samples_split,
    'min_samples_leaf': min_samples_leaf,
}

dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=0), dt_param_grid, cv=cvKFold)
dt_grid.fit(X_train, y_train)
dt_best_max_depth = dt_grid.best_params_['max_depth']
dt_best_min_samples_split = dt_grid.best_params_['min_samples_split']
dt_best_min_samples_leaf = dt_grid.best_params_['min_samples_leaf']
dt_cv_acc = dt_grid.best_score_
dt_test_acc = dt_grid.score(X_test, y_test)

In [14]:
# AdaBoost
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]
ada_param_grid = {'n_estimators': n_estimators, 'learning_rate': learning_rate}

ada_grid = GridSearchCV(AdaBoostClassifier(random_state=0), ada_param_grid, cv=cvKFold)
ada_grid.fit(X_train, y_train)
ada_best_n_estimators = ada_grid.best_params_['n_estimators']
ada_best_learning_rate = ada_grid.best_params_['learning_rate']
ada_cv_acc = ada_grid.best_score_
ada_test_acc = ada_grid.score(X_test, y_test)

In [15]:
# Gradient Boosting
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]
gb_param_grid = {
    'max_depth': max_depth,
    'n_estimators': n_estimators,
    'learning_rate': learning_rate,
}

gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=0), gb_param_grid, cv=cvKFold)
gb_grid.fit(X_train, y_train)
gb_best_max_depth = gb_grid.best_params_['max_depth']
gb_best_n_estimators = gb_grid.best_params_['n_estimators']
gb_best_learning_rate = gb_grid.best_params_['learning_rate']
gb_cv_acc = gb_grid.best_score_
gb_test_acc = gb_grid.score(X_test, y_test)

In [16]:
# Random Forest
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]
rf_param_grid = {'n_estimators': n_estimators, 'max_leaf_nodes': max_leaf_nodes}

rf_grid = GridSearchCV(
    RandomForestClassifier(criterion='entropy', max_features='sqrt', random_state=0),
    rf_param_grid,
    cv=cvKFold,
)
rf_grid.fit(X_train, y_train)
rf_best_n_estimators = rf_grid.best_params_['n_estimators']
rf_best_max_leaf_nodes = rf_grid.best_params_['max_leaf_nodes']
rf_cv_acc = rf_grid.best_score_
rf_test_acc = rf_grid.score(X_test, y_test)
rf_pred = rf_grid.predict(X_test)
rf_f1_macro = f1_score(y_test, rf_pred, average='macro')
rf_f1_weighted = f1_score(y_test, rf_pred, average='weighted')

In [17]:
# SVM
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]
svm_param_grid = {'C': C, 'gamma': gamma}

svm_grid = GridSearchCV(SVC(random_state=0), svm_param_grid, cv=cvKFold)
svm_grid.fit(X_train, y_train)
svm_best_C = svm_grid.best_params_['C']
svm_best_gamma = svm_grid.best_params_['gamma']
svm_cv_acc = svm_grid.best_score_
svm_test_acc = svm_grid.score(X_test, y_test)

### Part 2: Results

In [18]:
print("KNN best k: {}".format(knn_best_k))
print("KNN best p: {}".format(knn_best_p))
print("KNN cross-validation accuracy: {:.4f}".format(knn_cv_acc))
print("KNN test set accuracy: {:.4f}".format(knn_test_acc))

print("DT best max_depth: {}".format(dt_best_max_depth))
print("DT best min_samples_split: {}".format(dt_best_min_samples_split))
print("DT best min_samples_leaf: {}".format(dt_best_min_samples_leaf))
print("DT cross-validation accuracy: {:.4f}".format(dt_cv_acc))
print("DT test set accuracy: {:.4f}".format(dt_test_acc))

print("AdaBoost best n_estimators: {}".format(ada_best_n_estimators))
print("AdaBoost best learning_rate: {}".format(ada_best_learning_rate))
print("AdaBoost cross-validation accuracy: {:.4f}".format(ada_cv_acc))
print("AdaBoost test set accuracy: {:.4f}".format(ada_test_acc))

print("GB best max_depth: {}".format(gb_best_max_depth))
print("GB best n_estimators: {}".format(gb_best_n_estimators))
print("GB best learning_rate: {}".format(gb_best_learning_rate))
print("GB cross-validation accuracy: {:.4f}".format(gb_cv_acc))
print("GB test set accuracy: {:.4f}".format(gb_test_acc))

print("RF best n_estimators: {}".format(rf_best_n_estimators))
print("RF best max_leaf_nodes: {}".format(rf_best_max_leaf_nodes))
print("RF cross-validation accuracy: {:.4f}".format(rf_cv_acc))
print("RF test set accuracy: {:.4f}".format(rf_test_acc))
print("RF test set macro average F1: {:.4f}".format(rf_f1_macro))
print("RF test set weighted average F1: {:.4f}".format(rf_f1_weighted))

print("SVM best C: {}".format(svm_best_C))
print("SVM best gamma: {}".format(svm_best_gamma))
print("SVM cross-validation accuracy: {:.4f}".format(svm_cv_acc))
print("SVM test set accuracy: {:.4f}".format(svm_test_acc))

KNN best k: 5
KNN best p: 1
KNN cross-validation accuracy: 0.9381
KNN test set accuracy: 0.9286
DT best max_depth: 5
DT best min_samples_split: 5
DT best min_samples_leaf: 1
DT cross-validation accuracy: 0.9314
DT test set accuracy: 0.9400
AdaBoost best n_estimators: 150
AdaBoost best learning_rate: 0.5
AdaBoost cross-validation accuracy: 0.9438
AdaBoost test set accuracy: 0.9429
GB best max_depth: 1
GB best n_estimators: 50
GB best learning_rate: 0.3
GB cross-validation accuracy: 0.9448
GB test set accuracy: 0.9457
RF best n_estimators: 30
RF best max_leaf_nodes: 12
RF cross-validation accuracy: 0.9390
RF test set accuracy: 0.9371
RF test set macro average F1: 0.9355
RF test set weighted average F1: 0.9370
SVM best C: 5
SVM best gamma: 1
SVM cross-validation accuracy: 0.9457
SVM test set accuracy: 0.9343


## 3. Comparison Reference (fill in from your submitted notebook)Paste the values from the original (leaky) notebook below to get an automatic diff table.

In [19]:
# These are the numbers PRINTED BY YOUR SUBMITTED (leaky) notebook, hardcoded
# here for reference. If you re-run the leaky notebook and the numbers change
# (e.g. you edit the pipeline), update these values to match.
leaky = {
    "LogR (CV)": 0.9386,
    "NB (CV)": 0.9264,
    "KNN (test)": 0.9257,
    "DT (test)": 0.9400,
    "AdaBoost (test)": 0.9429,
    "GB (test)": 0.9457,
    "RF (test)": 0.9371,
    "SVM (test)": 0.9343,
}
noleak = {
    "LogR (CV)": logreg_acc,
    "NB (CV)": nb_acc,
    "KNN (test)": knn_test_acc,
    "DT (test)": dt_test_acc,
    "AdaBoost (test)": ada_test_acc,
    "GB (test)": gb_test_acc,
    "RF (test)": rf_test_acc,
    "SVM (test)": svm_test_acc,
}

print("{:16s} {:>10s} {:>10s} {:>10s}".format("Model", "Leaky", "No-leak", "Diff"))
for name in leaky:
    d = noleak[name] - leaky[name]
    if abs(d) < 5e-5:  # leaky values are hardcoded at 4dp, so sub-rounding noise isn't a real diff
        d = 0.0
    print("{:16s} {:>10.4f} {:>10.4f} {:>+10.4f}".format(name, leaky[name], noleak[name], d))

Model                 Leaky    No-leak       Diff
LogR (CV)            0.9386     0.9386    +0.0000
NB (CV)              0.9264     0.9264    +0.0000
KNN (test)           0.9257     0.9286    +0.0029
DT (test)            0.9400     0.9400    +0.0000
AdaBoost (test)      0.9429     0.9429    +0.0000
GB (test)            0.9457     0.9457    +0.0000
RF (test)            0.9371     0.9371    +0.0000
SVM (test)           0.9343     0.9343    +0.0000
